# Framework checkout

Click **Run All** for the current Git state, tool versions and basic hardware-free checks.
Each command prints its output below its cell. Failed commands remain visible and later
checks still run. Use your existing Linux Jupyter environment with Docker available.

This notebook creates no separate reports or exports. Pytest still creates its normal
framework test results/cache and may clean incompatible generated build state.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os
import platform
import shlex
import subprocess
import sys

# Auto-detect when opened from the repository or notebooks folder.
# Set this to the repository path if your Jupyter working directory is elsewhere.
REPO_ROOT = None
start = Path(REPO_ROOT or Path.cwd()).resolve()
root = next((p for p in (start, *start.parents)
             if (p / "AI_BUILD_GUIDE.md").is_file()
             and (p / "scripts/dev.sh").is_file()), None)
if root is None:
    raise RuntimeError("Set REPO_ROOT to your PQC-Framework-02 repository path.")

outcomes = []

def run(*args):
    """Show a command's live output and keep its exit status for the summary."""
    print("$ " + shlex.join(args), flush=True)
    try:
        with subprocess.Popen(args, cwd=root, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True,
                              encoding="utf-8", errors="replace", bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                code = process.wait()
            except KeyboardInterrupt:
                process.terminate()
                raise
    except OSError as exc:
        print(f"Could not run command: {exc}")
        code = None
    outcomes.append((shlex.join(args), code))
    print(f"\nExit status: {code}\n", flush=True)

print(f"Checked at: {datetime.now(timezone.utc).isoformat()}")
print(f"Repository: {root}")
print(f"Notebook Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")


Checked at: 2026-09-28T12:49:47.366422+00:00
Repository: /home/timk/rmit/PQC-Framework-02
Notebook Python: 3.14.6
Platform: Linux-7.0.0-34-generic-x86_64-with-glibc2.43


## Git state

Branch, commit, staged/unstaged/untracked files, and submodule revisions.


In [2]:
run("git", "status", "--short", "--branch", "--untracked-files=all", "--ignore-submodules=none")
run("git", "log", "-1", "--format=Commit: %H%nCommit date: %cI")
run("git", "submodule", "status", "--recursive")


$ git status --short --branch --untracked-files=all --ignore-submodules=none
## feature/A1-timing-dataset...origin/feature/A1-timing-dataset
 M README.md
 M docs/007_framework_architecture.md
 M notebooks/00_checkout.ipynb
 D scripts/checkout_report.py
 D tests/integration/test_checkout_report_in.py
 D tests/unit/test_checkout_report.py
?? tests/unit/test_checkout_notebook.py

Exit status: 0

$ git log -1 '--format=Commit: %H%nCommit date: %cI'
Commit: 5e9413a667d8eb21af0101d19f5f0c3ea293113b
Commit date: 2026-09-28T22:41:15+10:00

Exit status: 0

$ git submodule status --recursive
 e15476a21182d19b3ef826a5b88f09f94bce5805 Middlewares/Third_Party/wolfMQTT (v1.14.1-708-ge15476a)
 c2ac5902c80504e2a0bcdb246686138b3143da60 Middlewares/Third_Party/wolfSSL (heads/master)
 c9007d6f5232ff4cedc5b39144ee429c99cf4901 instruments/esp32_ina228/lib/SparkFun_INA2XX_Arduino_Library (v1.0.0-3-gc9007d6)
 d52eafe3e9303399fda15661f3d7bb8fe3d7eabc instruments/esp32_ina228/lib/SparkFun_INA2XX_Arduino_Librar

## Tools


In [3]:
run("git", "--version")
run("docker", "version", "--format", "Client: {{.Client.Version}}; server: {{.Server.Version}}")
run("docker", "image", "inspect", "--format", "Image: {{.Id}}",
    os.environ.get("PQC_DOCKER_IMAGE", "pqc-framework-stm32-build"))
run("./scripts/dev.sh", "python", "--version")
run("./scripts/dev.sh", "pytest", "--version")


$ git --version
git version 2.53.0

Exit status: 0

$ docker version --format 'Client: {{.Client.Version}}; server: {{.Server.Version}}'
Client: 29.8.1; server: 29.8.1

Exit status: 0

$ docker image inspect --format 'Image: {{.Id}}' pqc-framework-stm32-build
Image: sha256:c9442b5b1e3f187ba1eb3a69b440e0761e002a3d88b625e970e892d69b99a971

Exit status: 0

$ ./scripts/dev.sh python --version
Python 3.13.15

Exit status: 0

$ ./scripts/dev.sh pytest --version
pytest 9.1.1

Exit status: 0



## Basic repository checks


In [4]:
run("sh", "-n", "scripts/dev.sh")
run("git", "diff", "--check")
run("git", "diff", "--cached", "--check")
run("./scripts/dev.sh", "python", "scripts/check_dependency_pins.py")


$ sh -n scripts/dev.sh

Exit status: 0

$ git diff --check

Exit status: 0

$ git diff --cached --check

Exit status: 0

$ ./scripts/dev.sh python scripts/check_dependency_pins.py
Immutable dependency pin policy passed.

Exit status: 0



## Unit tests

These run in the framework Docker image and require no gateway or physical hardware.


In [5]:
run("./scripts/dev.sh", "pytest", "tests/unit", "-ra")


$ ./scripts/dev.sh pytest tests/unit -ra
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0
rootdir: /workspace
configfile: pytest.ini
collected 1118 items

tests/unit/test_application_timebase.py .........                        [  0%]
tests/unit/test_application_timebase_hw_support.py ......                [  1%]
tests/unit/test_artifacts.py ....                                        [  1%]
tests/unit/test_baremetal_benchmark.py ...                               [  1%]
tests/unit/test_baremetal_benchmark_allocator.py ..                      [  2%]
tests/unit/test_baremetal_benchmark_integration.py .....                 [  2%]
tests/unit/test_baremetal_benchmark_summary.py ..                        [  2%]
tests/unit/test_baseline_uart_evidence.py .........                      [  3%]
tests/unit/test_benchmark_aggregation.py ....                            [  3%]
tests/unit/test_build.py .......

## Summary

Exit status 0 means the command succeeded. Read pytest output for failures and skips.
These checks provide software status only; they do not establish hardware or runtime results.


In [6]:
for command, code in outcomes:
    status = "OK" if code == 0 else "UNAVAILABLE" if code is None else f"FAILED ({code})"
    print(f"{status:14} {command}")


OK             git status --short --branch --untracked-files=all --ignore-submodules=none
OK             git log -1 '--format=Commit: %H%nCommit date: %cI'
OK             git submodule status --recursive
OK             git --version
OK             docker version --format 'Client: {{.Client.Version}}; server: {{.Server.Version}}'
OK             docker image inspect --format 'Image: {{.Id}}' pqc-framework-stm32-build
OK             ./scripts/dev.sh python --version
OK             ./scripts/dev.sh pytest --version
OK             sh -n scripts/dev.sh
OK             git diff --check
OK             git diff --cached --check
OK             ./scripts/dev.sh python scripts/check_dependency_pins.py
OK             ./scripts/dev.sh pytest tests/unit -ra
